# Search lookup magics

Use **`%%cribl_save_search_lookup`** / **`%%cribl_load_search_lookup`** / **`%%cribl_delete_search_lookup`** for Search lookup CSV files under the **`default_search`** API group (same as `%%cribl_search`).

For a full **threat-hunting** walkthrough (Search dataset provider REST + KQL `join` + lookup lifecycle), see **`Threat_Hunting_Playbook.ipynb`**.

**Limits:** lookups are capped at **10,000 rows**. Use **`replace=true`** on save to overwrite. Pick **distinct filenames** (for example `notebook_app_lookup_demo.csv`) so you do not clash with shared lookups.

This notebook also shows **querying lookup contents** with the `$vt_lookups` virtual table (`lookupFile` is the **stem** without `.csv`; see Cribl [`$vt_lookups`](https://docs.cribl.io/search/vt_lookups/) docs), and **creating then deleting** a separate lookup with **`%%cribl_api`** (PUT upload + POST register + DELETE) so the request bodies use the same host `fetch` path as lookup magics.

## Setup

Pyodide already includes **pandas**. No `micropip` installs are required for this notebook—section **B** uses **`%%cribl_api`** (same host `fetch` path as lookup magics) instead of Python `pyfetch` so JSON bodies reach the control plane reliably.


In [ ]:
# No third-party installs required. Section B uses %%cribl_api for REST (host fetch + JSON body).
pass


## A) Cell magics: Search → save → search via `$vt_lookups` → load → verify


In [ ]:
%%cribl_search var=sample_df preview=false
dataset=cribl_search_sample | sort by _time desc | limit 50


In [ ]:
%%cribl_save_search_lookup notebook_app_lookup_demo.csv var=sample_df replace=true


In [ ]:
%%cribl_search var=vt_rows preview=true
dataset="$vt_lookups" lookupFile="notebook_app_lookup_demo"
| limit 25


In [ ]:
%%cribl_load_search_lookup notebook_app_lookup_demo.csv var=from_lookup


In [ ]:
print('search rows:', len(sample_df))
print('vt_lookups rows:', len(vt_rows))
print('loaded rows:', len(from_lookup))
from_lookup.head(3)


## B) Create & delete a lookup with `%%cribl_api`

The same **`%%cribl_api`** path the app uses for HTTP (host `fetch`, JSON serialization) avoids subtle **Pyodide worker `fetch` body** issues that can surface when calling **`pyfetch`** from Python for POST bodies.

Steps: **PUT** a small CSV as a temp upload, **POST** to register `notebook_app_sdk_lookup_demo.csv` from that temp file, then **DELETE** the registered lookup.


In [ ]:
%%cribl_api PUT /m/default_search/system/lookups?filename=__nb_sdk_tmp.csv var=nb_sdk_upload response=json
headers:
  Content-Type: text/csv
body: |
  join_key,label
  alpha,from_cribl_api
  beta,second_row



In [ ]:
%%cribl_api POST /m/default_search/system/lookups var=nb_sdk_register response=json template=on
json:
  id: "notebook_app_sdk_lookup_demo.csv"
  fileInfo:
    filename: "{{ nb_sdk_upload['filename'] }}"


In [ ]:
%%cribl_api DELETE /m/default_search/system/lookups/notebook_app_sdk_lookup_demo.csv var=nb_sdk_delete_result response=text


## Cleanup (cell magics)

Deletes the lookup created in section **A** (`notebook_app_lookup_demo.csv`). Section **B** removes `notebook_app_sdk_lookup_demo.csv` via `%%cribl_api` DELETE.


In [ ]:
%%cribl_delete_search_lookup notebook_app_lookup_demo.csv
